# Analyse the output of a multi-region retrieval

Same idea as `analyse_retrieval.ipynb` (load the run, choose the walkers, make figures from them) for a model with
several regions on the planet (e.g. hotspot and the rest, or day and night). It has **everything the single-region
notebook has** (chains, model, comparison with the low-resolution data, contribution of the species, profiles and
spectra confidence intervals), computed for the whole planet, **plus the views specific to multi-region**:
the rotation kernel of each region, and the comparison from one region to the other for the profiles, spectra,
synthetic data and species contributions.

In all the figures, the confidence intervals (1 and 2 sigma) are shown together with **individual curves** drawn from
the sample (`n_lines` of them), to see what the intervals are made of.

A region is drawn "alone" (what the planet would show if this region covered all of it: no kernel, no weight), and
"Global" is the combined spectrum of the whole planet, as fitted.

Sections: 1. Load the retrieval, 2. Walkers and selection, 3. Model for specific parameters (and save it for
`logl_grid`), 4. Comparison with low-resolution data, 5. Rotation kernels, 6. Profiles by region,
7. Contribution of each species, 8. Spectra confidence intervals, 9. Abundance ratios *(coming)*.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from starships import retrieval
from starships import retrieval_analysis as ra
import starships.retrieval_utils as ru
from starships.plotting_fcts import (get_plot_labels, scatterplot_logl, get_plot_limits_from_data,
                                     plot_tp_sample, plot_p_profile_sample, plot_profile_combined,
                                     setup_default_plot_params, plot_spectra_sample_GTC, plot_contributions, colorsDict)
from starships import petitradtrans_utils

petitradtrans_utils.log.setLevel('INFO')

# Load the retrieval

In [ ]:
# The yaml file of each run is saved in the retrieval_params directory (scratch), you can take any of them.
yaml_file = Path.home() / 'scratch/DataAnalysis/retrieval_params/KELT-20_b/params_emission_HRR_WL_wfc3_g395H_1_g395H_2_spitzer_modif_disso_3355174_1.yaml'

In [ ]:
# This loads all the setup of the retrieval (priors, data, instruments...) but does not run anything.
retrieval.setup_retrieval(yaml_file)

# What does this run allow us to do? The sections that need something the run does not have are skipped.
run = ra.describe_run(retrieval)
print(run.summary())

# Get labels
labels = get_plot_labels(params=retrieval.params_prior)

# Get walker output path
walker_path = retrieval.walker_path

print('\nDefault retrieval walker file:', retrieval.walker_file_out)

Note that the default walker file may not be a valid file (e.g. if it was set to None in the input yaml file, or the number at the end of the name). **The retrieval `walker_file` can be changed as long as it has the same parameters.**

In [ ]:
# Show the last walker files
walker_files_recent = sorted(walker_path.glob('*.h5'), key=lambda f: f.stat().st_mtime, reverse=True)
for f in walker_files_recent[:50]:
    print(f.name)

In [ ]:
if not run.is_multi_region:
    raise ValueError('This notebook is for multi-region runs (region_id with more than one region in the yaml file). '
                     'Use analyse_retrieval.ipynb instead.')

# One color set (pyGTC) and its darkest color per region, used in all the figures
region_color_sets = ['reds', 'blues', 'greens', 'oranges', 'purples'][:run.n_regions]
region_colors = [colorsDict[color_set][0] for color_set in region_color_sets]
region_names = [f'Region {region}' for region in run.region_ids]

# Color set of the whole planet ("Global"): the first one not used by a region
global_color_set = [c for c in ['greens', 'purples', 'oranges', 'yellows'] if c not in region_color_sets][0]
global_color = colorsDict[global_color_set][0]

# Number of individual curves (profiles, spectra, kernels) drawn on top of the confidence intervals
n_lines = 30

## Read the walker file

In [ ]:
walker_file = Path('walker_steps_emission_HRR_WL_wfc3_g395H_1_g395H_2_spitzer_modif_disso_3355174_burnin.h5')
# walker_file = retrieval.walker_file_out

# Number of steps to discard at the start of each walker (burn-in)
discard = 0

print('walker file:', walker_file)
chains = ru.read_walkers_file(walker_path / walker_file, discard=discard)

In [ ]:
chains.shape

In [ ]:
maxs, errors = ru.find_dist_maxs(chains, labels, bin_size=6, print_latex_style=True, alpha_plot=0.05)

In [ ]:
maxs2s, errors2s = ru.find_dist_maxs(chains, labels, bin_size=6, prob=0.954, plot=False, print_latex_style=True)

In [ ]:
fig, flat_samples = ru.plot_corner(chains, labels, param_no_zero=4,
                                   maxs=maxs, errors=errors, plot_datapoints=False)

In [ ]:
# Get the posterior distribution ordered by their logl value (last value is the highest logl)
ord_pos, ord_logl, autocorr = ra.load_posterior_sample(walker_path / walker_file, discard=discard)

In [ ]:
autocorr

## Verify the logl of each parameter
**This is useful if you want to do some analysis on burnin results.**

Based on the logl scatter plot (see below):
- For burnins, are there 2 clear blobs separated vertically (in logl value)?
- Is there an actual maximum?
- Are some walkers stuck in a secondary maximum (if so, we might remove them)?
- Do you want to restrain the parameters to a subsample for later use? Like to generate the spectra.

In [ ]:
print('Median logl value:', np.median(ord_logl))

prior_keys = list(retrieval.params_prior.keys())

In [ ]:
# Make a selection of the walkers if wanted

# On the logl (Set it to -np.inf if not used)
cond = ord_logl > 14994

# Or additional criteria on the parameters (uncomment and edit what you need).
# NOTE: for a parameter with a log-uniform prior (like the abundances), the value in the chains is its log10.
# idx = prior_keys.index('tp_gamma')
# cond &= ord_pos[:, idx] > 0.2

# For example on molecules
# idx = prior_keys.index('ptrans')
# cond &= ord_pos[:, idx] > -4.

# idx = prior_keys.index('log_f_wfc3')
# cond &= ord_pos[:, idx] < 1.5

# idx = prior_keys.index('kp')
# cond &= ord_pos[:, idx] < 200
# cond &= ord_pos[:, idx] > 190

# idx = prior_keys.index('rv')
# cond &= ord_pos[:, idx] > 10
# cond &= ord_pos[:, idx] < 20

# Regions can be selected separately, the parameters have the region number
# at the end of their name (e.g. T_eq_1, T_eq_2)
# idx = prior_keys.index('T_eq_1')
# cond &= ord_pos[:, idx] > 2000.

ord_pos_sub = ord_pos[cond]
ord_logl_sub = ord_logl[cond]
print(f'{cond.sum()} / {len(cond)} samples kept')

In [ ]:
fig, ax = scatterplot_logl(ord_pos, ord_logl, n_max_pts=10000, tight_ylim=True,
                           alpha=0.1, color='k', ylabels=labels)

# Comment/uncomment the following lines to plot the sub_samples
_ = scatterplot_logl(ord_pos_sub, ord_logl_sub, n_max_pts=10000, tight_ylim=True,
                     alpha=0.3, color='darkgreen', ylabels=labels, ax=ax)

# Zoom on one parameter if needed (the index is the parameter position in the list above)
# ax[2].set_xlim(-30, 30)

### You can also remake the corner plot for the subsample

In [ ]:
fig, flat_samples = ru.plot_corner(ord_pos_sub[:, None, :], labels, param_no_zero=4, plot_datapoints=False)

## Optional: compare with a CMA-ES best-fit (`run_starships_optimize`)
If you ran `run_starships_optimize` on this burn-in, its `.npz` result(s) hold a genuine
maximum-a-posteriori fit (only the maximum, no exploration of the rest of the posterior -- see
`starships/retrieval_optimize.py`'s module docstring), which can be a real amount higher than any
MCMC sample, plus every candidate CMA-ES evaluated along the way (`theta_trace`/`lnprob_trace`).

The dedicated notebook `tutorials/retrieval_optimize_examples/Inspect_optimize_trace.ipynb`
compares several starts against each other (no petitRADTRANS needed there, faster to iterate on) --
do that first if you ran more than one start, to check they agree before trusting one. Here, once
you know which one to trust, this section compares it to the MCMC chain above and lets you use its
best-fit for the model further down. **Optional**: leave `use_optimizer_result = False` to skip it.

In [ ]:
use_optimizer_result = False

# --- Edit this: the .npz result files, one per CMA-ES start ---
optimizer_result_files = [walker_path / f'optimum_{walker_file.stem}_start{i}.npz' for i in (1, 2, 3)]

opt_result = None
if use_optimizer_result:
    from starships.retrieval_optimize import compare_results

    compare_results(optimizer_result_files)  # same table as run_starships_optimize --compare

    # Picks the start with the highest lnprob_best -- edit to force a specific one instead
    # (e.g. chosen_file = optimizer_result_files[0]) if the comparison above suggests otherwise:
    # starts far from each other in lnprob mean the maximum is not reliable, see the printout.
    chosen_file = max(optimizer_result_files, key=lambda f: np.load(f)['lnprob_best'])
    opt_result = np.load(chosen_file)
    print(f'Using {chosen_file.name}')

    if list(opt_result['param_names']) != prior_keys:
        print('WARNING: parameter order/names differ between this yaml and the optimizer result -- '
             'best_params below would be wrong. Check you loaded the matching yaml file.')

In [ ]:
if opt_result is not None:
    n_trace, n_evals = len(opt_result['lnprob_trace']), int(opt_result['n_evals'])
    print(f"lnprob: burn-in best = {ord_logl.max():.2f}  |  optimizer start = "
         f"{float(opt_result['lnprob_start']):.2f}  |  optimizer best = {float(opt_result['lnprob_best']):.2f}")
    print(f"Trace covers {n_trace}/{n_evals} evaluations ({100 * n_trace / max(n_evals, 1):.0f}%) -- "
         "less than 100% for an old run resumed after the trace feature was added, see the quicklook "
         "notebook mentioned above.")

    # Convergence curve, with the MCMC's own best logl as a reference
    fig, ax = plt.subplots(figsize=(8, 4))
    history = opt_result['history']  # columns: n_evals, best lnprob so far, generation median, sigma
    ax.plot(history[:, 0], history[:, 1], label='Best so far (CMA-ES)')
    ax.plot(history[:, 0], history[:, 2], alpha=0.5, label='Generation median')
    ax.axhline(ord_logl.max(), color='k', linestyle='--', label='Best MCMC sample (this file)')
    ax.axhline(float(opt_result['lnprob_start']), color='grey', linestyle=':', label='Burn-in start point')
    ax.set_xlabel('Number of lnprob evaluations')
    ax.set_ylabel('lnprob')
    ax.legend(fontsize='small')

In [ ]:
if opt_result is not None:
    # Overlay the optimizer's search trace on the MCMC's own per-parameter logl scatter plot
    # (same scatterplot_logl tool the quicklook notebook uses to compare several starts to each
    # other, applied here instead to compare one start with the chain).
    fig, ax = scatterplot_logl(ord_pos, ord_logl, n_max_pts=10000, tight_ylim=True,
                              alpha=0.1, color='k', ylabels=labels)
    _ = scatterplot_logl(opt_result['theta_trace'], opt_result['lnprob_trace'], n_max_pts=10000,
                        tight_ylim=False, alpha=0.15, color='crimson', ylabels=labels, ax=ax)
    for ax_i, best_val in zip(ax, opt_result['theta_best']):
        ax_i.axvline(best_val, color='crimson', linestyle='--', linewidth=1)
    fig.suptitle('Black: MCMC chain -- Red: CMA-ES trace (dashed line = best-fit)', y=1.01)

# Change stellar spectrum to blackbody if wanted
Usually you don't want to do this, but it could be useful to differentiate the stellar signal from the planet's.
I did it personally to make plots for a presentation. If your stellar model is already a blackbody, then don't bother.

In [ ]:
from astropy.modeling import models
import astropy.units as u

# Use the temperature corresponding to the stellar spectrum (you can also play with it)
teff = 12000

if retrieval.star_spectrum is not None:

    fct_star = models.BlackBody(temperature=teff*u.K, scale=1*u.Unit('erg / (cm ** 2 * s * AA * sr)'))

    # Here I use a normalisation factor to make sure both models are agreeing.
    # The wavelength range is the range from WFC3 (the stellar spectrum is most important in spectrophotometry)
    wv = retrieval.star_wv
    in_range = (1.1 < wv) & (wv < 1.6)
    norm_factor = np.mean(fct_star(wv[in_range]*u.um)) / np.mean(retrieval.star_flux[in_range])

    def fct_star_BB(wv):
        return fct_star(wv*u.um) / norm_factor

    plt.plot(retrieval.star_wv[in_range], fct_star_BB(retrieval.star_wv)[in_range], label=f"scaled BB({teff}K)")

    plt.plot(retrieval.star_wv[in_range], retrieval.star_flux[in_range])
    plt.legend()

### Change the flux or not (comment if not)

In [ ]:
# retrieval.star_flux = fct_star_BB(retrieval.star_wv)

# Show the model for specific parameters

## Generate/initialize the model

### Choose which parameters you want (best logL?, median?)
TIP: Look at the logL plot to see which one represents the best your results.

In [ ]:
params_set = ord_pos_sub  # Sub sample defined above
# params_set = ord_pos  # full sample

# best_params = params_set[-1]  # Best logl
best_params = params_set[len(params_set) // 2]  # Median
# best_params = opt_result['theta_best']  # Use the CMA-ES best-fit instead (see the optional section above)

print(best_params)

### Convert the parameters array to a dictionary of parameters
`theta_regions` is a list of dictionaries with all the potential parameters that could be useful.
You can edit it here if you want before generating the model. Each dictionary represents a specific region
on the planet (so usually just one region).

In [ ]:
theta_regions = retrieval.unpack_theta(best_params)

In [ ]:
# Print the fitted parameters of each region
for region, theta_dict in zip(run.region_ids, theta_regions):
    print(f'--- Region {region}')
    for key in retrieval.params_prior:
        if key in theta_dict:
            print(key, theta_dict[key])

### Compute the spectra (the first time is slower due to initialization)
The whole planet (all the regions combined with their kernels) and each region alone. Same options as in the
single-region notebook: the high-resolution (lbl) model is computed if the run has high-resolution data, and can
be forced with `compute_high_res_model = True` (slow the first time).

In [ ]:
compute_high_res_model = run.can_model_high and run.has_high_res_data
# compute_high_res_model = True

# Instruments modelled at high resolution (lbl) need the high-res model too
need_high_res_model = compute_high_res_model or 'high' in run.low_res_instruments.values()

# Whole planet
wv_low, spec_low, _ = retrieval.prepare_static_model(theta_regions, 'low')
wv_high, spec_high = None, None
if need_high_res_model:
    wv_high, spec_high, _ = retrieval.prepare_static_model(theta_regions, 'high')

# Each region alone
region_models_low, region_models_high = {}, {}
for name, theta_dict in zip(region_names, theta_regions):
    wv_r, spec_r, _ = retrieval.prepare_static_model([theta_dict], 'low')
    region_models_low[name] = (wv_r, spec_r)
    if need_high_res_model:
        wv_r, spec_r, _ = retrieval.prepare_static_model([theta_dict], 'high')
        region_models_high[name] = (wv_r, spec_r)

In [ ]:
n_panels = 2 if spec_high is not None else 1
fig, ax = plt.subplots(n_panels, 1, figsize=(10, 4 * n_panels), squeeze=False)
ax = ax[:, 0]

for ax_i, models, kind in zip(ax, [region_models_low, region_models_high], ['c-k', 'lbl']):
    global_wv, global_spec = (wv_low, spec_low) if kind == 'c-k' else (wv_high, spec_high)
    ax_i.plot(global_wv, global_spec * 1e6, 'k', label='Global')
    for name, color in zip(region_names, region_colors):
        ax_i.plot(*models[name][:1], models[name][1] * 1e6, color=color, alpha=0.7, label=name)
    ax_i.set_ylabel(f'{retrieval.kind_trans} spectrum [ppm] ({kind})')
    ax_i.legend()
ax[-1].set_xlabel('Wavelength [um]')
plt.tight_layout()

### Save the model to give to `logl_grid`
The saved file has the native (line-by-line) planet and star spectra of each region and the parameters, so
`logl_grid` can redo exactly what the retrieval does (resolution of each visit, rotation kernels...).
It only makes sense for runs with high-resolution data. It uses `theta_regions` (not `best_params`), so the
edits you made above are saved. Give the file a name that says what selection it comes from.

In [ ]:
save_model_for_logl_grid = False
dir_models = Path.home() / 'scratch/starships_scratch/retrieval_analysis_tests/tmp_out'
model_file = dir_models / 'kelt20b_HRR_multi_region_test_model.npz'

if save_model_for_logl_grid:
    if not run.has_high_res_data:
        print('This run has no high-resolution data: there is nothing to give to logl_grid.')
    else:
        dir_models.mkdir(parents=True, exist_ok=True)
        planet_model = retrieval.save_planet_model(theta_regions, model_file)
        print(planet_model.describe())

In [ ]:
show_individual_species_high = False
save_individual_species_models = False

if (show_individual_species_high or save_individual_species_models) and run.can_model_high:
    for single_mol in retrieval.line_opacities:

        # Copy input dict
        theta_reg_single = [dict(theta_dict) for theta_dict in theta_regions]

        # Set all other linelists to zero (10^-99 to avoid division by zero). Continuum opacities
        # and other_species are untouched: the loop only ever assigns keys in line_opacities.
        for theta_dict in theta_reg_single:
            for mol in retrieval.line_opacities:
                if mol != single_mol:
                    theta_dict[mol] = 1e-99

        if show_individual_species_high:
            wv_i, spec_i, _ = retrieval.prepare_static_model(theta_reg_single, 'high')
            plt.plot(wv_i, spec_i, label=f'{single_mol} only', alpha=0.3)

        if save_individual_species_models:
            dir_models.mkdir(parents=True, exist_ok=True)
            species_model_file = dir_models / f'{model_file.stem}_{single_mol}_only.npz'
            retrieval.save_planet_model(theta_reg_single, species_model_file)
            print(f'Saved {species_model_file}')

    if show_individual_species_high:
        plt.legend()

## Quick comparison with low-resolution data
First for the whole planet (same figure as the single-region notebook), then with the binned model of each region
alone next to the data. All the data available in the yaml file is plotted, but it may not have been included in the
fit (e.g. in HRR mode).

In [ ]:
# Uncomment to change some settings of the low-res data before the comparison
# retrieval.spectrophotometric_data['g395H_1']['res'] = 900
# retrieval.spectrophotometric_data['g395H_2']['res'] = 900
# retrieval.photometric_data['spitzer']['wave'][1] *= 1.0001

In [ ]:
# Make synthetic data for all low-res instruments: the whole planet, then each region alone
synt_data_by_name = {'Global': ru.get_syntetic_data_low_res(wv_low, spec_low, wv_high, spec_high,
                                                            retrieval_obj=retrieval)}
for name in region_names:
    wv_r, spec_r = region_models_low[name]
    wv_rh, spec_rh = region_models_high.get(name, (None, None))
    synt_data_by_name[name] = ru.get_syntetic_data_low_res(wv_r, spec_r, wv_rh, spec_rh, retrieval_obj=retrieval)

synt_data_dict = synt_data_by_name['Global']

### Whole planet

In [ ]:
n_subplots = len(retrieval.spectrophotometric_data) + 1
fig, ax = plt.subplots(n_subplots, 1, figsize=(6, 3*n_subplots), squeeze=False)
ax = ax[:, 0]

# Plot the model in each panel and set labels
for ax_i in ax:
    if spec_high is not None:
        ax_i.plot(wv_high, spec_high * 1e6, label='lbl')
    ax_i.plot(wv_low, spec_low * 1e6, label="c-k")
    ax_i.set_ylabel(f"{retrieval.kind_trans} spectrum [ppm]")

# Make one panel per spectrophotometric instrument name
for i_instru, (instru_name, infos) in enumerate(retrieval.spectrophotometric_data.items()):

    x_plot, y_plot, yerr = infos['wave'], infos['data'], infos['err']
    y_plot = y_plot * 1e6  # in ppm
    yerr = yerr * 1e6  # in ppm

    # Force the plot limits around the low-res data
    xlim = get_plot_limits_from_data(x_plot)
    ylim = get_plot_limits_from_data(y_plot, pad=0.2)

    # Plots
    ax[i_instru].errorbar(x_plot, y_plot, yerr=yerr, fmt=".", label=instru_name)
    ax[i_instru].plot(x_plot, synt_data_dict[instru_name]*1e6, '.', label='Model binned')
    ax[i_instru].set_xlim(xlim)
    ax[i_instru].set_ylim(ylim)

    # Also plot in the last panel
    ax[-1].errorbar(x_plot, y_plot, yerr=yerr, fmt=".", label=instru_name)

    # Labels
    ax[i_instru].set_title(f"Zoom on {instru_name} data", loc='left')
    ax[i_instru].legend()


# Put everything in the last panel
for instru_name, infos in retrieval.photometric_data.items():

    x_plot, y_plot, yerr, x_limits = (infos[key] for key in ['wave', 'data', 'err', 'wv_coverages'])
    y_plot = y_plot * 1e6  # in ppm
    yerr = yerr * 1e6  # in ppm

    xerr = np.abs(np.transpose(x_limits) - x_plot)
    ax[-1].errorbar(x_plot, y_plot, yerr=yerr, xerr=xerr, fmt=".", label=instru_name)
    ax[-1].plot(x_plot, synt_data_dict[instru_name]*1e6, '.', label=f'Synt data ({instru_name})')


ax[-1].set_title(f"All data", loc='left')
ax[-1].set_xlabel("Wavelength [um]")
ax[-1].legend()
plt.tight_layout()

### Each region alone (binned model) next to the data

In [ ]:
entry_colors = {'Global': 'k', **dict(zip(region_names, region_colors))}

n_subplots = len(retrieval.spectrophotometric_data) + 1
fig, ax = plt.subplots(n_subplots, 1, figsize=(8, 3.5 * n_subplots), squeeze=False)
ax = ax[:, 0]

for i_instru, (instru_name, infos) in enumerate(retrieval.spectrophotometric_data.items()):
    x_plot, y_plot, yerr = infos['wave'], infos['data'] * 1e6, infos['err'] * 1e6

    for ax_i in [ax[i_instru], ax[-1]]:
        ax_i.errorbar(x_plot, y_plot, yerr=yerr, fmt='.', color='gray', label=instru_name)
    for name, synt in synt_data_by_name.items():
        ax[i_instru].plot(x_plot, synt[instru_name] * 1e6, '.', color=entry_colors[name], label=f'{name} (binned)')

    ax[i_instru].set_xlim(get_plot_limits_from_data(x_plot))
    ax[i_instru].set_ylim(get_plot_limits_from_data(y_plot, pad=0.2))
    ax[i_instru].set_title(f'Zoom on {instru_name} data', loc='left')
    ax[i_instru].set_ylabel(f'{retrieval.kind_trans} spectrum [ppm]')
    ax[i_instru].legend(fontsize='small')

for instru_name, infos in retrieval.photometric_data.items():
    x_plot, y_plot, yerr, x_limits = (infos[key] for key in ['wave', 'data', 'err', 'wv_coverages'])
    xerr = np.abs(np.transpose(x_limits) - x_plot)
    ax[-1].errorbar(x_plot, y_plot * 1e6, yerr=yerr * 1e6, xerr=xerr, fmt='.', color='gray', label=instru_name)
    for name, synt in synt_data_by_name.items():
        ax[-1].plot(x_plot, synt[instru_name] * 1e6, 'o', color=entry_colors[name], label=f'{name} ({instru_name})')

# In the last panel, all the data and the low-res spectrum of each entry
for name, (wv_r, spec_r) in {'Global': (wv_low, spec_low), **region_models_low}.items():
    ax[-1].plot(wv_r, spec_r * 1e6, color=entry_colors[name], alpha=0.5, lw=0.8)
ax[-1].set_title('All data', loc='left')
ax[-1].set_xlabel('Wavelength [um]')
ax[-1].set_ylabel(f'{retrieval.kind_trans} spectrum [ppm]')
ax[-1].legend(fontsize='small')
plt.tight_layout()

## Choose the sample used for all the confidence intervals
(Could be the selection above, or the full sample.)

In [ ]:
# flatten_sample = flat_samples
# flatten_sample = ord_pos
flatten_sample = ord_pos_sub

# Rotation kernels of each region
The kernel of each region is computed by the `get_ker` function of the run (`get_ker_file` in the yaml file), for
`n_draw_kernels` samples, exactly as the retrieval does: it includes the instrumental resolution and the weight of each
region at the chosen orbital phase (visibility), so the kernels of all the regions add up to 1.

- **Figure 1**: confidence intervals (line = median, bands = 1 and 2 sigma) at each phase.
- **Figure 2**: `n_lines` individual kernels drawn from the sample, one row per region and one for the sum.

In [ ]:
n_draw_kernels = 100
kernel_phases = [0.4, 0.5, 0.6]  # orbital phases where the kernels are computed (0.5 = secondary eclipse)

kernels_by_phase = {}
for phase in kernel_phases:
    velocity, kernels_by_phase[phase] = ra.draw_rotation_kernels(n_draw_kernels, flatten_sample, retrieval,
                                                                 phase=phase, seed=0)

# Zoom where the kernels are not zero
all_kernels = np.concatenate(list(kernels_by_phase.values()))
v_max = np.abs(velocity[all_kernels.sum(axis=(0, 1)) > 1e-3 * all_kernels.sum(axis=1).max()]).max()

In [ ]:
# Figure 1: confidence intervals
fig, axes = plt.subplots(1, len(kernel_phases), figsize=(5 * len(kernel_phases), 4), sharey=True, squeeze=False)
axes = axes[0]

for ax_i, phase in zip(axes, kernel_phases):
    kernels = kernels_by_phase[phase]

    for i_reg in range(run.n_regions):
        stats = ru.get_stats_from_profile(kernels[:, i_reg, :])
        ax_i.plot(velocity, stats['median'], color=region_colors[i_reg], label=region_names[i_reg])
        ax_i.fill_between(velocity, *stats['1-sig'], color=region_colors[i_reg], alpha=0.3)
        ax_i.fill_between(velocity, *stats['2-sig'], color=region_colors[i_reg], alpha=0.15)

    # The sum of the regions is what the spectrum is convolved with
    total_stats = ru.get_stats_from_profile(kernels.sum(axis=1))
    ax_i.plot(velocity, total_stats['median'], 'k--', label='Sum of regions')

    ax_i.set_xlim(-1.2 * v_max, 1.2 * v_max)
    ax_i.set_title(f'Phase {phase}')
    ax_i.set_xlabel('Velocity [km/s]')

axes[0].set_ylabel('Kernel')
axes[0].legend(fontsize='small')
plt.tight_layout()

In [ ]:
# Figure 2: individual kernels (one row per region + one for the sum)
n_rows = run.n_regions + 1
fig, axes = plt.subplots(n_rows, len(kernel_phases), figsize=(5 * len(kernel_phases), 2.5 * n_rows),
                         sharex=True, sharey=True, squeeze=False)

for i_col, phase in enumerate(kernel_phases):
    kernels = kernels_by_phase[phase]
    for i_reg in range(run.n_regions):
        axes[i_reg, i_col].plot(velocity, kernels[:n_lines, i_reg, :].T, color=region_colors[i_reg], alpha=0.2)
    axes[-1, i_col].plot(velocity, kernels[:n_lines].sum(axis=1).T, color='k', alpha=0.2)
    axes[0, i_col].set_title(f'Phase {phase}')
    axes[-1, i_col].set_xlabel('Velocity [km/s]')

for ax_row, label in zip(axes[:, 0], region_names + ['Sum of regions']):
    ax_row.set_ylabel(label)
axes[0, 0].set_xlim(-1.2 * v_max, 1.2 * v_max)
plt.tight_layout()

# Profiles by region
The same random samples are used for all the regions, so the regions are comparable. Abundances are computed as in
the retrieval (free chemistry, with dissociation if it was included).

In [ ]:
n_draw_profiles = 100
pressures, profile_samples_by_region = ra.draw_region_profiles(n_draw_profiles, flatten_sample, retrieval, seed=1)
profile_stats_by_region = [ra.compute_profile_stats(samples) for samples in profile_samples_by_region]

# `profile_samples_by_region[i]` has all the species that the model uses + the temperature profile of region i
all_species = [key for key in profile_samples_by_region[0] if key != 'temperature']

In [ ]:
def add_region_legend(ax, handles, names, xpos=0.1, ypos=0.9, d_ypos=-0.1):
    """Write the region names (in the color of their confidence interval) in an empty axes."""
    ax.axis('off')
    for handle, name in zip(handles, names):
        ax.text(xpos, ypos, name, color=handle.get_facecolor()[0], fontsize=16, transform=ax.transAxes)
        ypos += d_ypos

## Temperature profiles

In [ ]:
fig, ax = plt.subplots(1, 3, sharey=True, figsize=(12, 5), gridspec_kw=dict(width_ratios=[3, 3, 1]))

# Confidence intervals of each region
_, _, handles = plot_profile_combined(pressures, [stats['temperature'] for stats in profile_stats_by_region],
                                      ax=ax[0], colorsOrder=region_color_sets)
ax[0].set_title('Confidence intervals (1 and 2 sigma)', fontsize='small')

# A sample of the profiles
for samples, color in zip(profile_samples_by_region, region_colors):
    ax[1].semilogy(samples['temperature'][:n_lines].T, pressures, color=color, alpha=0.2)
ax[1].set_title(f'{n_lines} random profiles', fontsize='small')
ax[1].set_ylim(ax[0].get_ylim())

for ax_i in ax[:2]:
    ax_i.set_xlabel('Temperature [K]')
add_region_legend(ax[2], handles, region_names)
plt.tight_layout()

## Abundance profiles, species by species
Top: confidence intervals of each region. Bottom: individual profiles.

In [ ]:
# Species to compare (all the fitted ones by default)
species_compared = [mol for mol in retrieval.line_opacities + retrieval.continuum_opacities
                    if mol in all_species]
# species_compared = ['H2O', 'CO', 'Fe']

n_species = len(species_compared)
fig, ax = plt.subplots(2, n_species + 1, sharey=True, figsize=(3 * n_species + 2, 8), squeeze=False)

for i_col, specie in enumerate(species_compared):
    _, _, handles = plot_profile_combined(pressures, [stats[specie] for stats in profile_stats_by_region],
                                          ax=ax[0, i_col], colorsOrder=region_color_sets)
    for samples, color in zip(profile_samples_by_region, region_colors):
        ax[1, i_col].loglog(samples[specie][:n_lines].T, pressures, color=color, alpha=0.2)
    ax[1, i_col].set_ylim(ax[0, i_col].get_ylim())

    for ax_i in ax[:, i_col]:
        ax_i.set_xscale('log')
        ax_i.set_xlim(1e-12, 1e0)
        ax_i.set_ylabel(None)
    ax[0, i_col].set_title(specie)

add_region_legend(ax[0, -1], handles, region_names)
ax[1, -1].axis('off')
for ax_i in ax[:, 0]:
    ax_i.set_ylabel('Pressure [bar]', fontsize='large')
ax[1, n_species // 2].set_xlabel('Volume Mixing Ratio', fontsize='large')
plt.tight_layout()

## One figure per region
Temperature and abundance profiles of each region (confidence intervals and individual profiles).

In [ ]:
species_panels = [['H2O', 'H-', 'OH'], ['H-', 'e-'], ['H', 'H2']]

for region_name, stats, samples, color in zip(region_names, profile_stats_by_region,
                                              profile_samples_by_region, region_colors):
    fig, ax = plt.subplots(2, 1 + len(species_panels), sharey='row', figsize=(10, 8))

    # Top: confidence intervals
    plot_tp_sample(pressures, stats['temperature'], ax=ax[0, 0])
    for ax_i, mol_names in zip(ax[0, 1:], species_panels):
        for mol_name in mol_names:
            if mol_name in stats:
                plot_p_profile_sample(pressures, stats[mol_name], ax=ax_i, label=mol_name)
        ax_i.set_xscale('log')
        ax_i.set_xlim(1e-12, 1e0)
        ax_i.legend()

    # Bottom: individual profiles
    ax[1, 0].semilogy(samples['temperature'][:n_lines].T, pressures, 'k', alpha=0.2)
    ax[1, 0].set_xlabel('Temperature [K]')
    for ax_i, mol_names in zip(ax[1, 1:], species_panels):
        for i_mol, mol_name in enumerate(mol_names):
            if mol_name in samples:
                ax_i.loglog(samples[mol_name][:n_lines].T, pressures, color=f'C{i_mol}', alpha=0.2)
        ax_i.set_xlim(1e-12, 1e0)
    ax[1, 0].set_ylim(ax[0, 0].get_ylim())
    ax[1, 2].set_xlabel('Volume Mixing Ratio', fontsize='large')
    fig.suptitle(region_name)
    plt.tight_layout()

## Compare with chemical equilibrium, region by region
Needs petitRADTRANS. `c_to_o` is fixed by hand, and the metallicity is estimated from the free-chemistry Fe/H
ratio if `Fe` is fitted (otherwise `metallicity`, 0 = solar). NOTE: `metallicity` here is petitRADTRANS's
equilibrium-chemistry knob, NOT literally the elemental Fe/H ratio (Fe/H is only used as its standard proxy).

In [ ]:
compare_chemical_equilibrium = True

if compare_chemical_equilibrium:
    c_to_o = .95
    metallicity = None

    profile_stats_eq_by_region = []
    for samples in profile_samples_by_region:
        profile_samples_eq = ra.equilibrium_profiles(samples, pressures, c_to_o=c_to_o, metallicity=metallicity)
        profile_stats_eq_by_region.append(ra.compute_profile_stats(profile_samples_eq))

    for region_name, stats, stats_eq in zip(region_names, profile_stats_by_region, profile_stats_eq_by_region):
        species_shared = [specie for specie in stats if specie in stats_eq and specie != 'temperature']
        n_shared = len(species_shared)
        fig, ax = plt.subplots(1, n_shared + 1, sharey=True, figsize=(3 * n_shared, 4))

        for ax_i, specie in zip(ax, species_shared):
            _, _, handles = plot_profile_combined(pressures, [stats_eq[specie], stats[specie]], ax=ax_i)
            ax_i.set_xscale('log')
            ax_i.set_title(specie)
            ax_i.set_ylabel(None)

        add_region_legend(ax[-1], handles, ['Equilibrium', 'Free'])
        ax[0].set_ylabel('Pressure [bar]', fontsize='large')
        fig.suptitle(region_name)

# Contribution of each species
For the parameters chosen above (`theta_regions`). The regions are computed as if each one covered the whole planet;
"Global" is the combined spectrum, as fitted. Each figure is separate:

1. one figure per region, and one for the whole planet (the same 6 panels as in the single-region notebook),
2. one figure per species, comparing the regions and the whole planet,
3. *(optional, slow)* confidence intervals and individual curves of the contribution of each species.

In [ ]:
species_contrib = retrieval.line_opacities  # or a subset, e.g. ['H2O', 'CO', 'Fe']
mode_contrib = 'low'  # 'high' is much slower

theta_by_name = {name: [theta] for name, theta in zip(region_names, theta_regions)}
theta_by_name['Global'] = theta_regions

contrib = {name: ru.get_contribution(species_contrib, retrieval, theta_list, mode=mode_contrib)
           for name, theta_list in theta_by_name.items()}

### 1. One figure per region and for the whole planet

In [ ]:
for name, (wv_c, spec_dict) in contrib.items():
    fig, ax = plot_contributions(wv_c, spec_dict, retrieval.kind_trans, figsize=(10, 16), title=name)

### 2. One figure per species
Top: the full spectrum, the spectrum without the species and the species alone. Bottom: the contribution of the species,
measured two ways (solid: full minus without the species, dashed: the species alone minus the spectrum with no species).

In [ ]:
n_cols = len(contrib)

for specie in species_contrib:
    fig, ax = plt.subplots(2, n_cols, sharex='col', sharey='row', figsize=(5 * n_cols, 7), squeeze=False)

    for i_col, (name, (wv_c, spec_dict)) in enumerate(contrib.items()):
        ax[0, i_col].plot(wv_c, spec_dict['All'] * 1e6, label='All species')
        ax[0, i_col].plot(wv_c, spec_dict[f'Without {specie}'] * 1e6, label=f'Without {specie}')
        ax[0, i_col].plot(wv_c, spec_dict[f'{specie} only'] * 1e6, label=f'{specie} only', alpha=0.6)
        ax[1, i_col].plot(wv_c, (spec_dict['All'] - spec_dict[f'Without {specie}']) * 1e6,
                          label=f'All - without {specie}')
        ax[1, i_col].plot(wv_c, (spec_dict[f'{specie} only'] - spec_dict['No Species']) * 1e6, '--',
                          label=f'{specie} only - no species')
        ax[0, i_col].set_title(name)
        ax[1, i_col].set_xlabel('Wavelength [um]')

    ax[0, 0].set_ylabel(f'{retrieval.kind_trans} spectrum [ppm]')
    ax[1, 0].set_ylabel('Contribution [ppm]')
    ax[0, 0].legend(fontsize='x-small')
    ax[1, 0].legend(fontsize='x-small')
    fig.suptitle(specie)
    plt.tight_layout()

### 3. Confidence intervals of the contribution of each species -- optional, slow
`n_draw_contrib` samples, each needing 2 spectra per species for each region and the whole planet. One figure per
species: the individual contributions (thin lines), the median and the 1 sigma interval, for each region and the
whole planet.

In [ ]:
contributions_with_ci = False
n_draw_contrib = 20

if contributions_with_ci:
    waves_c, spectra_c = ra.draw_spectra(n_draw_contrib, flatten_sample, retrieval, kind_res=mode_contrib,
                                         species_list=species_contrib, by_region=True, seed=3)
    entries = [f'region_{region}' for region in run.region_ids] + ['global']
    entry_titles = region_names + ['Global']
    entry_line_colors = region_colors + [global_color]

    for specie in species_contrib:
        fig, ax = plt.subplots(1, len(entries), sharey=True, figsize=(5 * len(entries), 3.5), squeeze=False)
        for ax_i, name, title, color in zip(ax[0], entries, entry_titles, entry_line_colors):
            sp = spectra_c[name]
            alone = (sp[f'{specie} only'] - sp['No Species']) * 1e6   # (n_draw, n_wave)
            ax_i.plot(waves_c[name], alone[:n_lines].T, color=color, alpha=0.2, lw=0.5)
            stats = ru.get_stats_from_profile(alone)
            ax_i.fill_between(waves_c[name], *stats['1-sig'], color=color, alpha=0.4)
            ax_i.plot(waves_c[name], stats['median'], 'k', lw=1)
            ax_i.set_title(title)
            ax_i.set_xlabel('Wavelength [um]')
        ax[0, 0].set_ylabel(f'{specie} contribution [ppm]')
        fig.suptitle(specie)
        plt.tight_layout()

# Spectra statistics

## First, we need to generate the spectra from the sample

In [ ]:
n_draw = 50
kind_res = 'low'  # 'high' takes much longer

# The whole planet ('global') and each region alone, from the same draws
waves, spec_samples_by_name = ra.draw_spectra(n_draw, flatten_sample, retrieval, kind_res=kind_res,
                                              by_region=True, seed=2)
spec_stats_by_name = {name: ru.get_stats_from_profile(samples['All'])
                      for name, samples in spec_samples_by_name.items()}

## Then generate the synthetic data for each low res instrument

In [ ]:
# Instruments modelled at high resolution (lbl) need the line-by-line spectra of the same draws
waves_high, spec_samples_high_by_name = None, None
if 'high' in run.low_res_instruments.values():
    waves_high, spec_samples_high_by_name = ra.draw_spectra(n_draw, flatten_sample, retrieval, kind_res='high',
                                                            by_region=True, seed=2)

synt_data_stats_by_name = {}
for name, samples in spec_samples_by_name.items():
    high = None if waves_high is None else spec_samples_high_by_name[name]['All']
    synt_data = ra.synthetic_data_samples(samples['All'], retrieval, wave_low=waves[name], spec_samples_high=high,
                                          wave_high=None if waves_high is None else waves_high[name])
    synt_data_stats_by_name[name] = {instru: ru.get_stats_from_profile(sample) for instru, sample in synt_data.items()}

## The spectra of each region and of the whole planet
Confidence intervals with individual spectra drawn from the sample.

In [ ]:
setup_default_plot_params()

In [ ]:
scale_factor = 100  # Scale the spectrum to change units (100 -> units in percent)
ylabel = r'F$_p$ / F$_{\star}$ [\%]'

# One curve per region, then the whole planet
entries = [f'region_{region}' for region in run.region_ids] + ['global']
entry_titles = region_names + ['Global']
entry_color_sets = region_color_sets + [global_color_set]
entry_line_colors = region_colors + [global_color]

# One column per entry. Top: confidence intervals. Bottom: individual spectra drawn from the sample.
fig, ax = plt.subplots(2, len(entries), sharex=True, sharey=True, figsize=(5.5 * len(entries), 8), squeeze=False)
for i_col, (name, title, color_set, color) in enumerate(zip(entries, entry_titles, entry_color_sets,
                                                            entry_line_colors)):
    plot_spectra_sample_GTC(waves[name], [spec_stats_by_name[name]], colorsOrder=[color_set],
                            ax=ax[0, i_col], scale_factor=scale_factor)
    ax[1, i_col].plot(waves[name], spec_samples_by_name[name]['All'][:n_lines].T * scale_factor,
                      color=color, alpha=0.3, lw=0.5)
    ax[0, i_col].set_title(title)
    ax[1, i_col].set_xlabel(r'Wavelength [$\mu$m]')
ax[0, 0].set_ylabel(ylabel)
ax[1, 0].set_ylabel(ylabel)
plt.tight_layout()

In [ ]:
# All the entries on the same axes (confidence intervals only)
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
for name, color_set in zip(entries, entry_color_sets):
    plot_spectra_sample_GTC(waves[name], [spec_stats_by_name[name]], colorsOrder=[color_set],
                            ax=ax, scale_factor=scale_factor)
for legend_name, color in zip(entry_titles, entry_line_colors):
    ax.plot([], [], color=color, label=legend_name)
ax.set_xlabel(r'Wavelength [$\mu$m]')
ax.set_ylabel(ylabel)
ax.legend(fontsize=12)

## Synthetic data confidence intervals versus the data
First the whole planet (same figure as the single-region notebook), then the whole planet and each region together.

In [ ]:
# Variables used by the single-region figure below
wv = waves['global']
spec_stats = {'All': spec_stats_by_name['global']}
synt_data_stats = {'All': synt_data_stats_by_name['global']}

In [ ]:
save_fig = False

specie = 'All'
scale_factor = 100  # Scale the spectrum to change units (100 -> units in percent)
ylabel = r'F$_p$ / F$_{\star}$ [\%]'
spec_stats_colors = 'greens'  # Color set from pyGTC

# You can add some plotting infos to the specific instruments (like the color!)
# 'wv_offset': To shift statistics in the plot so they don't overlap with the datapoints (better display)
instrument_plot_infos = {
    'spitzer': dict(data_color='purple', stats_color='deeppink', wv_offset=0.05),
}
for instru_name, plot_infos in instrument_plot_infos.items():
    for data_dict in [retrieval.spectrophotometric_data, retrieval.photometric_data]:
        if instru_name in data_dict:
            data_dict[instru_name].update(plot_infos)


# ==============
# Do the plot
# ==============

n_subplots = len(retrieval.spectrophotometric_data) + 1
fig, ax = plt.subplots(n_subplots, 1, figsize=(10, 5*n_subplots), squeeze=False)
ax = ax[:, 0]

# Plot the model in each panel and set labels
for ax_i in ax:
    if spec_high is not None:
        ax_i.plot(wv_high, spec_high * scale_factor, 'lightblue', label='High res', zorder=-1)
    ax_i.plot(wv_low, spec_low * scale_factor, label="Low res")
    ax_i.set_ylabel(ylabel)

# Plot the full model stats in the last panel
stats_plot = spec_stats[specie]
plot_spectra_sample_GTC(wv, [stats_plot], colorsOrder=[spec_stats_colors],
                        ax=ax[-1], scale_factor=scale_factor)

# Make one panel per spectrophotmetric instrument name
for i_instru, (instru_name, infos) in enumerate(retrieval.spectrophotometric_data.items()):

    # Get the observation
    x_plot, y_plot, yerr = infos['wave'], infos['data'], infos['err']
    y_plot = y_plot * scale_factor  # scaled to plot units
    yerr = yerr * scale_factor  # scaled to plot units

    # Get the stats
    stats_plot = synt_data_stats[specie][instru_name]

    # Force the plot limits around the low-res data
    xlim = get_plot_limits_from_data(x_plot)
    ylim = get_plot_limits_from_data(y_plot, pad=0.2)

    # Infos for the plot, specific to instrument
    data_color = infos.get('data_color', None)

    # Plots
    plot_spectra_sample_GTC(x_plot, [stats_plot], colorsOrder=[spec_stats_colors],
                            ax=ax[i_instru], scale_factor=scale_factor)
    ax[i_instru].errorbar(x_plot, y_plot, yerr=yerr, fmt=".", color=data_color, label=instru_name)
    ax[i_instru].set_xlim(xlim)
    ax[i_instru].set_ylim(ylim)

    # Also plot the instrument data in the last panel
    ax[-1].errorbar(x_plot, y_plot, yerr=yerr, fmt=".", color=data_color, label=instru_name)

    # Labels
    ax[i_instru].set_title(f"Zoom on {instru_name} data", loc='left')
    ax[i_instru].legend(fontsize=12)


# Put everything in the last panel
for instru_name, infos in retrieval.photometric_data.items():

    x_plot, y_plot, yerr, x_limits = (infos[key] for key in ['wave', 'data', 'err', 'wv_coverages'])
    y_plot = y_plot * scale_factor  # scaled to plot units
    yerr = yerr * scale_factor  # scaled to plot units

    # Get the stats
    stats_plot = synt_data_stats[specie][instru_name]

    # Infos for the plot, specific to instrument
    data_color = infos.get('data_color', None)
    stats_color = infos.get('stats_color', None)
    x_offset = infos.get('wv_offset', 0.)

    # Common plotting arguments
    comm_kwargs = dict(linestyle='', marker='', linewidth=2)

    # Plot observed data
    xerr = np.abs(np.transpose(x_limits) - x_plot)
    ax[-1].errorbar(x_plot, y_plot, yerr=yerr, xerr=xerr,
                    color=data_color, label=instru_name, **comm_kwargs)

    # Plot synthetic data stats (1 and 2 sigma)
    yerr = np.abs(stats_plot['2-sig'] - stats_plot['median'][None, :])
    ax[-1].errorbar(x_plot - x_offset, stats_plot['median'] * scale_factor,
                   yerr=yerr * scale_factor, color=stats_color,
                   capsize=8, capthick=2, **comm_kwargs)

    yerr = np.abs(stats_plot['1-sig'] - stats_plot['median'][None, :])
    ax[-1].errorbar(x_plot - x_offset, stats_plot['median'] * scale_factor,
                   yerr=yerr * scale_factor, color=stats_color,
                   capsize=4, capthick=2, **comm_kwargs)

ax[-1].set_xlabel(r'Wavelength [$\mu$m]')
ax[-1].set_ylabel(ylabel)
ax[-1].legend(fontsize=12)

fig_filename = Path(walker_file.stem.replace('walker_steps', 'spectra')).with_suffix('.pdf')
if save_fig:
    fig_filename = Path('Spectra_profiles') / fig_filename
    fig_filename.parent.mkdir(exist_ok=True)
    print('Saving to', fig_filename)
    fig.savefig(fig_filename)

### The whole planet and each region together

In [ ]:
# The first entry is drawn on top
plot_entries = ['global'] + [f'region_{region}' for region in run.region_ids]
plot_color_sets = [global_color_set] + region_color_sets
plot_colors = [global_color] + region_colors
plot_titles = ['Global'] + region_names

n_subplots = len(retrieval.spectrophotometric_data) + 1
fig, ax = plt.subplots(n_subplots, 1, figsize=(10, 5 * n_subplots), squeeze=False)
ax = ax[:, 0]

for i_instru, (instru_name, infos) in enumerate(retrieval.spectrophotometric_data.items()):
    x_plot, y_plot, yerr = infos['wave'], infos['data'] * scale_factor, infos['err'] * scale_factor

    plot_spectra_sample_GTC(x_plot, [synt_data_stats_by_name[name][instru_name] for name in plot_entries],
                            colorsOrder=plot_color_sets, ax=ax[i_instru], scale_factor=scale_factor)
    for ax_i in [ax[i_instru], ax[-1]]:
        ax_i.errorbar(x_plot, y_plot, yerr=yerr, fmt='.', color='gray', label=instru_name)
    ax[i_instru].set_xlim(get_plot_limits_from_data(x_plot))
    ax[i_instru].set_ylim(get_plot_limits_from_data(y_plot, pad=0.2))
    ax[i_instru].set_title(f'Zoom on {instru_name} data', loc='left')
    ax[i_instru].set_ylabel(ylabel)
    ax[i_instru].legend(fontsize=12)

# Last panel: the spectra of each entry, the data, and the photometry
for name, color_set in zip(plot_entries, plot_color_sets):
    plot_spectra_sample_GTC(waves[name], [spec_stats_by_name[name]], colorsOrder=[color_set],
                            ax=ax[-1], scale_factor=scale_factor)

for instru_name, infos in retrieval.photometric_data.items():
    x_plot, y_plot, yerr, x_limits = (infos[key] for key in ['wave', 'data', 'err', 'wv_coverages'])
    xerr = np.abs(np.transpose(x_limits) - x_plot)
    ax[-1].errorbar(x_plot, y_plot * scale_factor, yerr=yerr * scale_factor, xerr=xerr,
                    color='gray', label=instru_name, linestyle='', marker='')

    # Synthetic data of each entry (1 and 2 sigma), shifted in wavelength so they do not overlap
    for j, (name, color) in enumerate(zip(plot_entries, plot_colors)):
        stats_plot = synt_data_stats_by_name[name][instru_name]
        x_shift = (j - (len(plot_entries) - 1) / 2) * 0.05
        for key, capsize in [('2-sig', 8), ('1-sig', 4)]:
            yerr = np.abs(stats_plot[key] - stats_plot['median'][None, :])
            ax[-1].errorbar(x_plot + x_shift, stats_plot['median'] * scale_factor, yerr=yerr * scale_factor,
                            color=color, capsize=capsize, capthick=2, linestyle='', marker='')

for color, title in zip(plot_colors, plot_titles):
    ax[-1].plot([], [], color=color, label=title)
ax[-1].set_xlabel(r'Wavelength [$\mu$m]')
ax[-1].set_ylabel(ylabel)
ax[-1].legend(fontsize=12)
plt.tight_layout()

# Abundance ratios
*Coming next:* ratios of abundances (C/O, [X/H] relative to solar...) as a function of pressure, for each region, with
confidence intervals, and compared at pressures chosen by the user.